In [1]:
import unicodedata
from sqlalchemy import text
from common import *
from database import get_engine

dfs = load_raw()

def clean_text(s):
    f = lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode()
    return s.astype("string").str.strip().str.lower().map(f, na_action="ignore")

# ---- customers, sellers, geolocation: text + zip standardisation ----
customers, sellers, geo = dfs["customers"], dfs["sellers"], dfs["geolocation"]
for df, city, state, zc in [
    (customers, "customer_city", "customer_state", "customer_zip_code_prefix"),
    (sellers, "seller_city", "seller_state", "seller_zip_code_prefix"),
    (geo, "geolocation_city", "geolocation_state", "geolocation_zip_code_prefix"),
]:
    df[city] = clean_text(df[city])
    df[state] = df[state].str.strip().str.upper()
    df[zc] = df[zc].str.zfill(5)

# geolocation: ~1M rows -> one row per zip prefix
geo = (geo.groupby("geolocation_zip_code_prefix")
          .agg(geolocation_lat=("geolocation_lat", "mean"),
               geolocation_lng=("geolocation_lng", "mean"),
               geolocation_city=("geolocation_city", lambda x: x.mode().iat[0]),
               geolocation_state=("geolocation_state", lambda x: x.mode().iat[0]))
          .reset_index())

# ---- orders: dates + logical validity ----
orders = dfs["orders"].copy()
for c in orders.columns[3:]:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")
orders["order_status"] = orders["order_status"].str.strip().str.lower()
bad = orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]
print("impossible delivery dates nulled:", bad.sum())
orders.loc[bad, "order_delivered_customer_date"] = pd.NaT
orders = orders[orders["customer_id"].isin(customers["customer_id"])]

# ---- products + category translation ----
prod = dfs["products"].rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"})
prod["product_category_name"] = prod["product_category_name"].fillna("unknown")
num = prod.select_dtypes("number").columns
prod[num] = prod[num].fillna(prod[num].median())

tr = dfs["product_category_translation"]
missing = set(prod["product_category_name"]) - set(tr["product_category_name"])
tr = pd.concat([tr, pd.DataFrame({"product_category_name": list(missing),
                                  "product_category_name_english": list(missing)})],
               ignore_index=True)

# ---- order items ----
items = dfs["order_items"].copy()
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")
items = items[(items["price"] > 0) & (items["freight_value"] >= 0)]
items = items[items["order_id"].isin(orders["order_id"])
              & items["product_id"].isin(prod["product_id"])
              & items["seller_id"].isin(sellers["seller_id"])]

# ---- payments ----
pay = dfs["order_payments"].copy()
pay["payment_type"] = pay["payment_type"].str.strip().str.lower()
pay = pay[(pay["payment_type"] != "not_defined") & (pay["payment_installments"] >= 1)]
pay = pay[pay["order_id"].isin(orders["order_id"])]

# ---- reviews: keep the latest review per order ----
rev = dfs["order_reviews"].copy()
for c in ["review_creation_date", "review_answer_timestamp"]:
    rev[c] = pd.to_datetime(rev[c], errors="coerce")
rev = rev[rev["review_score"].between(1, 5)]
rev = (rev.sort_values("review_answer_timestamp")
          .drop_duplicates("order_id", keep="last"))
rev = rev[rev["order_id"].isin(orders["order_id"])]

cleaned = {
    "geolocation": geo, "customers": customers.drop_duplicates(),
    "sellers": sellers.drop_duplicates(), "product_category_translation": tr,
    "products": prod.drop_duplicates(), "orders": orders.drop_duplicates(),
    "order_items": items.drop_duplicates(), "order_payments": pay.drop_duplicates(),
    "order_reviews": rev,
}
for name, df in cleaned.items():
    df.to_csv(CLEAN / f"{name}.csv", index=False)
    print(f"{name:32s} raw={len(dfs[name]):>8,}  cleaned={len(df):>8,}")

impossible delivery dates nulled: 0
geolocation                      raw=1,000,163  cleaned=  19,015
customers                        raw=  99,441  cleaned=  99,441
sellers                          raw=   3,095  cleaned=   3,095
product_category_translation     raw=      71  cleaned=      74
products                         raw=  32,951  cleaned=  32,951
orders                           raw=  99,441  cleaned=  99,441
order_items                      raw= 112,650  cleaned= 112,650
order_payments                   raw= 103,886  cleaned= 103,881
order_reviews                    raw=  99,224  cleaned=  98,673


In [2]:
engine = get_engine()
load_order = list(cleaned)          # parents first (dict order above is FK-safe)

with engine.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS=0"))
    for t in reversed(load_order):
        conn.execute(text(f"DROP TABLE IF EXISTS {t}"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS=1"))
    for stmt in (ROOT / "sql" / "schema.sql").read_text().split(";"):
        if stmt.strip():
            conn.execute(text(stmt))

for t in load_order:
    cleaned[t].to_sql(t, engine, if_exists="append", index=False,
                      chunksize=2000, method="multi")

# Verify
with engine.connect() as conn:
    for t, df in cleaned.items():
        n = conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
        print(f"{t:32s} pandas={len(df):>8,}  mysql={n:>8,}  {'OK' if n == len(df) else 'MISMATCH'}")

geolocation                      pandas=  19,015  mysql=  19,015  OK
customers                        pandas=  99,441  mysql=  99,441  OK
sellers                          pandas=   3,095  mysql=   3,095  OK
product_category_translation     pandas=      74  mysql=      74  OK
products                         pandas=  32,951  mysql=  32,951  OK
orders                           pandas=  99,441  mysql=  99,441  OK
order_items                      pandas= 112,650  mysql= 112,650  OK
order_payments                   pandas= 103,881  mysql= 103,881  OK
order_reviews                    pandas=  98,673  mysql=  98,673  OK
